# El problema del viatjant
Tens huit ciutats. Busca una ruta curta que passe per totes i torne al punt de partida. Una ruta és una tupla d'índexs, com `(0, 2, 1, …)`. Mantindrem la ciutat 0 al principi.

Aplicarem l’escalada a una ruta inicial. En cada pas, compararem rutes que diferisquen en un únic intercanvi i ens quedarem amb la més curta si millora l’actual.

In [ ]:
from itertools import combinations, permutations
from math import dist, exp
from random import Random

rng = Random(1)
ciutats = [(rng.randrange(100), rng.randrange(100)) for _ in range(8)]
inicial = tuple(range(len(ciutats)))

def cost(ruta):
    circuit = ruta + (ruta[0],)
    return sum(dist(ciutats[a], ciutats[b]) for a, b in zip(circuit, circuit[1:]))

def intercanvia(ruta, i, j):
    nova = list(ruta)
    nova[i], nova[j] = nova[j], nova[i]
    return tuple(nova)

def veines(ruta):
    for i, j in combinations(range(1, len(ruta)), 2):
        yield intercanvia(ruta, i, j)

## El primer pas de l’escalada
Compara el cost de la ruta inicial amb el del millor veí. Si el cost baixa, faries aquest intercanvi.

In [ ]:
millor_vei = min(veines(inicial), key=cost)
print(inicial, round(cost(inicial), 2))
print(millor_vei, round(cost(millor_vei), 2))

## Hill climbing
En cada pas intercanviem les posicions de dues ciutats. Provem els veïns i ens quedem amb el millor si redueix el cost. Si cap intercanvi millora la ruta, hem arribat a un mínim local.

In [ ]:
def hill_climbing(ruta):
    while True:
        millor = min(veines(ruta), key=cost, default=ruta)
        if cost(millor) >= cost(ruta):
            return ruta
        ruta = millor

local = hill_climbing(inicial)
print(local, round(cost(local), 2))

## És la millor ruta possible?
Quan l’escalada s’atura, cap intercanvi millora la ruta. Per saber si hi ha un circuit més curt, comparem totes les 7! rutes que comencen en 0. Amb huit ciutats és viable. El nombre de rutes creix ràpidament quan afegim ciutats.

In [ ]:
optima = min(((0,) + resta for resta in permutations(inicial[1:])), key=cost)
print(optima, round(cost(optima), 2))
print(round(cost(local) - cost(optima), 2))  # Diferència de cost

## Variants de la cerca local
Amb **reinicis aleatoris** repetim hill climbing des de rutes diferents i guardem la millor. Amb **recuit simulat** provem un intercanvi aleatori cada vegada. Si empitjora el cost en `delta`, l'acceptem amb probabilitat `exp(-delta / temperatura)`. Refredem a cada pas i guardem la millor ruta que hem vist.

**Primer millor** prova els veïns en ordre aleatori i tria el primer que millora la ruta.

In [ ]:
def primer_millor(ruta, rng):
    while True:
        candidates = list(veines(ruta))
        rng.shuffle(candidates)
        for candidata in candidates:
            if cost(candidata) < cost(ruta):
                ruta = candidata
                break
        else:
            return ruta

def amb_reinicis(ruta, rng, intents=10):
    millor = hill_climbing(ruta)
    for _ in range(intents):
        nova = (ruta[0],) + tuple(rng.sample(ruta[1:], len(ruta) - 1))
        resultat = hill_climbing(nova)
        if cost(resultat) < cost(millor):
            millor = resultat
    return millor

def recuit(ruta, rng, passos=500, temperatura=30):
    millor = ruta
    for _ in range(passos):
        i, j = rng.sample(range(1, len(ruta)), 2)
        candidata = intercanvia(ruta, i, j)
        delta = cost(candidata) - cost(ruta)
        if delta <= 0 or rng.random() < exp(-delta / temperatura):
            ruta = candidata
            if cost(ruta) < cost(millor):
                millor = ruta
        temperatura *= 0.99
    return millor

In [ ]:
primer = primer_millor(inicial, Random(42))
reinicis = amb_reinicis(inicial, Random(42))
recuit_final = recuit(inicial, Random(42))
for nom, ruta in (("Inicial", inicial), ("Hill climbing", local),
                  ("Primer millor", primer),
                  ("Reinicis / Reinicios", reinicis),
                  ("Recuit / Recocido", recuit_final), ("Òptim / Óptimo", optima)):
    print(f"{nom}: {cost(ruta):.2f}  {ruta}")

Canvia la llavor dels reinicis o la temperatura del recuit. Arribes sempre al mateix cost que amb la cerca exacta?

## Visualitza les rutes
Compara el circuit que dona l’escalada amb el circuit òptim. El dibuix inclou l’últim tram, que torna a la ciutat 0.

In [ ]:
from IPython.display import HTML, display

def dibuixa_ruta(ruta, titol):
    punts = [(20 + 2*x, 220 - 2*y) for x, y in ciutats]
    circuit = ruta + (ruta[0],)
    linia = " ".join(f"{punts[i][0]},{punts[i][1]}" for i in circuit)
    svg = [f'<svg viewBox="0 0 240 260" width="240" role="img" aria-label="{titol}">',
           f'<polyline points="{linia}" fill="none" stroke="#2668a3" stroke-width="2"/>']
    for i, (x, y) in enumerate(punts):
        svg.append(f'<circle cx="{x}" cy="{y}" r="4" fill="#2668a3"/>')
        svg.append(f'<text x="{x + 5}" y="{y - 5}" fill="#444">{i}</text>')
    svg.append(f'<text x="16" y="248" fill="#444">{titol} · {cost(ruta):.2f}</text></svg>')
    return "".join(svg)

display(HTML(dibuixa_ruta(local, "Hill climbing") + dibuixa_ruta(optima, "Òptim")))